In [2]:
from datasets import load_dataset

dataset = load_dataset("flwrlabs/ucf101")

# To see what's inside (columns, number of rows, etc.)
print(dataset)

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/79 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/31 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'video_id', 'clip_id', 'frame', 'label'],
        num_rows: 1786096
    })
    test: Dataset({
        features: ['image', 'video_id', 'clip_id', 'frame', 'label'],
        num_rows: 697222
    })
})


In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from helpers import downsize_clip

## --- 1. SETUP FUNCTIONS --- ##

def downsize_clip(frames, fps):
    """
    Downsize a video clip to the specified frames per second (fps).
    
    Parameters:
    frames (list): A list of frames from the video clip.
    fps (int): The desired frames per second for the downsized clip.
    """
    step = int(30 / fps)  
    clip_frames = frames[::step]   
    return clip_frames  


def load_qwen_model(model_id="Qwen/Qwen2.5-VL-3B-Instruct"):
    """Initialize the model and processor."""
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_id, torch_dtype="auto", device_map="auto"
    )
    processor = AutoProcessor.from_pretrained(model_id)
    return model, processor

## --- 2. DATA STRATEGY FUNCTIONS --- ##

def get_clip_generator(dataset_split, max_rows=1000):
    """
    Efficiently yields one full clip at a time using contiguous indexing.
    """
    current_row = 0
    total_len = len(dataset_split)
    
    while current_row < max_rows and current_row < total_len:
        first_row = dataset_split[current_row]
        clip_id = first_row["clip_id"]
        label_id = first_row["label"]
        
        # Resolve action name from metadata
        label_name = dataset_split.features["label"].int2str(label_id)
        
        clip_images = []
        # Fast inner loop for contiguous frames
        while current_row < total_len:
            row = dataset_split[current_row]
            if row["clip_id"] == clip_id:
                clip_images.append(row["image"])
                current_row += 1
            else:
                break
        
        yield {
            "clip_id": clip_id,
            "label_name": label_name,
            "images": clip_images
        }

## --- 3. INFERENCE ENGINE --- ##

def run_inference(model, processor, images, prompt="Describe the action in this video."):
    """Handles the Qwen-VL conversation and generation logic."""
    
    # 1. Downsize using your helper
    downsized = downsize_clip(images, fps=1)
    
    # 2. Format Messages
    messages = [{
        "role": "user",
        "content": [
            {"type": "video", "video": downsized, "fps": 1.0},
            {"type": "text", "text": prompt},
        ],
    }]

    # 3. Process Inputs
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    # 4. Generate
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=64)
        
    # Trim the prompt from the output
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    return processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)[0]



In [11]:
import time

# Initialize tracking variables
total_processing_time = 0.0
total_frames_processed = 0.0

# Initialize Model
model, processor = load_qwen_model()
test_data = dataset["test"]

print("🚀 Starting Pipeline...")

for clip in get_clip_generator(test_data, max_rows=1000):
    print(f"\n🎬 Clip: {clip['clip_id']} | Expected: {clip['label_name']}")
    
    # --- Start Timer ---
    start_time = time.time()
    
    # Run inference
    response = run_inference(model, processor, clip['images'])
    
    # --- End Timer ---
    end_time = time.time()
    
    # Calculate stats for this specific clip
    # Note: We use the length of the downsized frames for the 'per frame' calculation
    # since those are what the model actually "saw"
    num_frames = len(clip['images']) # Or use len(downsized) inside run_inference if preferred
    duration = end_time - start_time
    
    total_processing_time += duration
    total_frames_processed += num_frames
    
    avg_time_per_frame = duration / num_frames if num_frames > 0 else 0
    
    print(f"🤖 Qwen: {response}")
    print(f"⏱️  Clip Time: {duration:.2f}s | Avg Time per Frame: {avg_time_per_frame:.4f}s")
    
    
    
    # Clear memory
    torch.cuda.empty_cache()

# Global average update
global_avg = total_processing_time / total_frames_processed
print(f"📊 Running Global Average: {global_avg:.4f}s per frame")

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

🚀 Starting Pipeline...

🎬 Clip: v_ApplyEyeMakeup_g01_c01 | Expected: ApplyEyeMakeup
🤖 Qwen: A woman is applying makeup to her eyes with a brush. She is looking down at her face and moving the brush back and forth across her eyelids.
⏱️  Clip Time: 3.37s | Avg Time per Frame: 0.0205s

🎬 Clip: v_ApplyEyeMakeup_g01_c02 | Expected: ApplyEyeMakeup
🤖 Qwen: A woman is applying makeup to her eyes with a brush. She is looking down at her face and moving the brush across her eyelids.
⏱️  Clip Time: 2.24s | Avg Time per Frame: 0.0182s

🎬 Clip: v_ApplyEyeMakeup_g01_c03 | Expected: ApplyEyeMakeup
🤖 Qwen: A woman is applying makeup to her eyes with a brush. She is looking directly at the camera and appears to be demonstrating how to apply the product.
⏱️  Clip Time: 2.90s | Avg Time per Frame: 0.0112s

🎬 Clip: v_ApplyEyeMakeup_g01_c04 | Expected: ApplyEyeMakeup
🤖 Qwen: A woman is applying makeup to her eyes with a brush. She is looking directly at the camera and smiling.
⏱️  Clip Time: 2.27s | Avg T